# 3. Models and resultsThe ladder, each step fixing a weakness in the one before, then two models that gobeyond the standard recipe:* **M4** matrix factorisation with a **censored loss**, because ~70% of DAVIS labels are  a detection limit rather than a measurement.* **M5/M6** **KronRLS**, Kronecker kernel ridge, which answers all four regimes with one  model because the only thing that changes is which kernel it is handed.

In [ ]:
import sys, warningssys.path.insert(0, "../src")warnings.filterwarnings("ignore")import numpy as np, pandas as pdpd.set_option("display.width", 180); pd.set_option("display.max_columns", 50)from dta import data as D, splits as S, models as M, metrics as MT, kernels as K, plots as Pfrom dta.kronrls import KronRLSfrom dta.config import DATASETSP.use_style()summary = pd.read_csv("../results/tables/summary_all.csv")

## Warm start (S1)

In [ ]:
for ds in DATASETS:    d = summary[(summary.dataset==ds) & (summary.setting=="S1_pair")]    cols = ["model","mse_mean","mse_strong_mean","ci_mean","ci_per_ligand_mean",            "hit_at_1_mean","random_hit_at_1_mean"]    print(f"=== {ds.upper()}"); display(d[[c for c in cols if c in d.columns]].round(4))P.warm_comparison(summary)

## The censored-loss experimentThe prediction, stated in the model's docstring before it was run: training against adetection limit as if it were a measurement should buy MSE and cost ranking. So thecensored model should look **worse** on MSE and **better** on ranking.On KIBA, which has no censoring, it must collapse onto squared error exactly. That is acorrectness check as much as a result.

In [ ]:
for ds in DATASETS:    d = summary[(summary.dataset==ds)&(summary.setting=="S1_pair")].set_index("model")    if "M4_tobit_mf" not in d.index: continue    print(f"=== {ds.upper()}")    for m in ["mse_mean","mse_strong_mean","ci_mean","ci_per_ligand_mean","mrr_mean"]:        if m in d.columns:            print(f"  {m:22s} censored {d.loc['M4_tobit_mf',m]:.4f}   squared {d.loc['M4b_tobit_squared',m]:.4f}")P.censored_effect(summary)

## All four regimes, one modelPer-ligand CI is the metric that matters: it ranks a drug's candidate targets, which isthe use case, and it is the one that exposes cold-start failure. A model with noinformation about an unseen target scores every candidate alike, so every ranking is atie and the value is exactly 0.5.

In [ ]:
for ds in DATASETS:    print(f"=== {ds.upper()}")    d = summary[summary.dataset==ds]    display(d.pivot_table(index="model", columns="setting",                          values="ci_per_ligand_mean").round(3))    P.four_settings(summary, ds)

### Why global CI would have hidden itThe same fallback scores 0.70 to 0.78 on pooled CI, because the drug biases still orderdrugs sensibly within any one target. Reporting only pooled CI would make a model thatcannot rank at all look respectable.

In [ ]:
for ds in DATASETS:    d = summary[(summary.dataset==ds)&(summary.setting=="S3_new_target")].set_index("model")    if "C0_fallback" not in d.index: continue    print(f"{ds.upper()}: fallback  global CI {d.loc['C0_fallback','ci_mean']:.3f}   "          f"per-ligand CI {d.loc['C0_fallback','ci_per_ligand_mean']:.3f}")